# OntologyRAG-Q — best configuration + Chain-of-Thought, run with Gemma

Runs the best-performing setup from Table 4 of Al-Azani et al. (EMNLP 2025):

**Ayat-Ontology chunking · k = 6 · Similarity · temperature = 0.0 · multilingual-e5-small**

with Gemma as the generator, and **chain-of-thought (CoT) prompting** added on top of
the retrieval step. The last cells print your result as a Table 4 row and then show
worked examples in English.

---

### What changed from the previous notebook

| | Before | Now |
|---|---|---|
| Prompting | answer directly | **CoT**: reason step by step, then answer |
| Eval size | 200 | **500** (`N_EVAL_SAMPLES`) |
| Output shown | Arabic only | Arabic **+ English translation** |
| Answer scored | whole generation | **only the `ANSWER:` block** |

### Why the answer is still Arabic

The reference answers in `OntologyQA_v1.json` are Arabic. BLEU, CHRF and BERTScore
compare your output against those references, so the scored answer has to be Arabic
or the numbers are meaningless and no longer comparable to the paper.

So the split is: **reasoning in English, final answer in Arabic, and an English
translation printed alongside everything you read.** Nothing Arabic is shown to you
without a translation next to it.

---

### Setup on Kaggle (do this first)

1. Go to https://huggingface.co/google/gemma-3-4b-it and click **Acknowledge license**.
2. Go to https://huggingface.co/settings/tokens and create a token, type **Read**. Copy it.
3. In this notebook: **Add-ons → Secrets → Add a new secret**. Label it `HF_TOKEN`, paste the token, tick the checkbox.
4. **Settings → Accelerator → GPU**.
5. Use the **same Hugging Face account** for steps 1 and 2.

### How to run

Leave `QUICK_TEST = True` and press **Run All**. It answers **5 questions** so you can
check everything works. Read the worked examples in the last cell — that is the check.

Then set `QUICK_TEST = False` and Run All again. Those same 5 questions are already
cached, so it continues straight on to the remaining 495.

If it stops early, just Run All again — it continues from where it stopped.

> **Runtime warning.** CoT generates reasoning *and* an answer, so each question costs
> roughly twice what it did before. On a T4 at 4-bit, budget **6–9 hours for all 500**.
> Kaggle caps a session at 9 hours, so you will likely need two or three Run Alls.
> The resume cache makes that safe. If it is too slow, switch `MODEL_ID` to
> `google/gemma-3-1b-it` or drop `N_EVAL_SAMPLES`.

In [ ]:
# ================= CONFIG =================
QUICK_TEST = True                       # True = 5 questions. Set False for the real run.

N_EVAL_SAMPLES = 500                    # the real run. None = all 2,350 Direct questions
QUICK_N = 5                             # how many QUICK_TEST answers
SEED = 42

MODEL_ID = "google/gemma-3-4b-it"       # or "google/gemma-3-1b-it" if the T4 is too slow
QUANT_BITS = 4                          # 8 = better quality, may not fit on a T4

# ---- Chain-of-Thought ----
# True  = model reasons step by step in English, then gives the Arabic answer.
#         Only the ANSWER block is scored; the reasoning is stored for inspection.
# False = the previous behaviour, for an A/B comparison. Results are written to
#         separate files, so you can run both and compare without overwriting.
USE_COT = True

# ---- English translation of the Arabic ----
# The answers STAY ARABIC -- that is the correct output and it is what gets
# scored against the Arabic references. But every Arabic string this notebook
# prints is followed by its English translation on "# EN:" comment lines, so you
# can always read what the model actually said. Translations are display only
# and never touch the metrics.
#
# Translating costs one extra generation per string, so the full run only
# translates a sample. In QUICK_TEST everything is translated.
TRANSLATE_N = None          # None = all of them in QUICK_TEST, 10 in the full run
TRANSLATE_PASSAGES = None   # None = translate retrieved passages in QUICK_TEST only

# Which books to search.
#   "all"    -- all 15 Tafsir books (55,471 chunks). Harder search.
#   "source" -- only the 2 books the answers were written from (~7,500 chunks).
#
# The paper's Limitations says "we only performed the analysis using one
# source", which is ambiguous. If they searched one book, their search was
# far easier than searching all 15, which would explain part of their high
# scores. Running both settings tells you how much the corpus size matters.
CORPUS = "all"

# The paper's best row -- do not change these.
TOP_K = 6
EMBED_MODEL_ID = "intfloat/multilingual-e5-small"

MAX_CHARS_PER_CHUNK = 2000
MAX_PROMPT_TOKENS = 8192
# CoT needs room for the reasoning AND the answer. Too small and the answer is
# cut off before it is ever written, which scores as an empty prediction.
MAX_NEW_TOKENS = 900 if USE_COT else 512
BERTSCORE_MODEL = "bert-base-multilingual-cased"
RESULTS_DIR = "results"

N_ANSWER = QUICK_N if QUICK_TEST else N_EVAL_SAMPLES
RUN_TAG = "cot" if USE_COT else "direct"
if TRANSLATE_N is None:
    TRANSLATE_N = N_ANSWER if QUICK_TEST else 10
if TRANSLATE_PASSAGES is None:
    TRANSLATE_PASSAGES = QUICK_TEST

if QUICK_TEST:
    print(f"QUICK_TEST on: {QUICK_N} questions. These are the first {QUICK_N} of the "
          f"same {N_EVAL_SAMPLES}-question sample, so the full run reuses them.")
print(f"{MODEL_ID} | {QUANT_BITS}-bit | k={TOP_K} | corpus={CORPUS} "
      f"| prompting={RUN_TAG} | n={N_ANSWER}")
print(f"Answers stay in Arabic; {TRANSLATE_N} of them get an English translation "
      f"printed as '# EN:' comments.")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(RESULTS_DIR, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"Could not read HF_TOKEN: {type(e).__name__}")
    print("Fix: Add-ons > Secrets > new secret labelled exactly HF_TOKEN, checkbox ticked.")

!pip install -q -U transformers accelerate bitsandbytes faiss-cpu sentence-transformers openpyxl sacrebleu bert-score tqdm

In [ ]:
# ========== CHECK EVERYTHING BEFORE THE SLOW STEPS ==========
import torch
ok = True

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"[OK]   GPU: {p.name}, {p.total_memory/1e9:.1f} GB")
else:
    ok = False
    print("[FAIL] No GPU. Settings > Accelerator > GPU, then restart the session.")

token = os.environ.get("HF_TOKEN")
if not token:
    ok = False
    print("[FAIL] HF_TOKEN missing -- see the cell above.")
else:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    try:
        print(f"[OK]   Token belongs to: {api.whoami().get('name')}")
    except Exception:
        ok = False
        print("[FAIL] Token rejected -- create a fresh token of type 'Read'.")
    try:
        api.model_info(MODEL_ID)
        print(f"[OK]   Access to {MODEL_ID} confirmed.")
    except Exception:
        ok = False
        print(f"[FAIL] No access to {MODEL_ID}.")
        print(f"       Open https://huggingface.co/{MODEL_ID}, click 'Acknowledge license',")
        print("       logged in as the user printed above.")

print("\nReady to run." if ok else "\nFix the [FAIL] items before continuing.")

In [ ]:
# ========== LOAD THE MODEL ==========
# Gemma-3-4B is a multimodal checkpoint and needs Gemma3ForConditionalGeneration.
# Gemma-3-1B is text-only and needs a plain causal-LM class. Try each in turn.
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

# A T4 is Turing (compute 7.5) and has NO native bfloat16 -- bf16 there is emulated
# and markedly slower. That cost is worth avoiding now that CoT doubles the token
# count. Pick bf16 only on Ampere (8.0) and newer, fp16 otherwise.
_major = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
COMPUTE_DTYPE = torch.bfloat16 if _major >= 8 else torch.float16
print(f"Compute dtype: {str(COMPUTE_DTYPE).split('.')[-1]} (GPU compute capability {_major}.x)")

bnb = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,
                          bnb_4bit_quant_type="nf4") if QUANT_BITS == 4 else
       BitsAndBytesConfig(load_in_8bit=True) if QUANT_BITS == 8 else None)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

classes = []
for name in ["Gemma3ForConditionalGeneration", "AutoModelForImageTextToText"]:
    try:
        classes.append(getattr(__import__("transformers", fromlist=[name]), name))
    except (ImportError, AttributeError):
        pass
classes.append(AutoModelForCausalLM)

model, errs = None, []
for cls in classes:
    try:
        model = cls.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
        print(f"Loaded with {cls.__name__} ({QUANT_BITS}-bit).")
        break
    except Exception as e:
        errs.append(f"{cls.__name__}: {type(e).__name__}: {str(e)[:200]}")

if model is None:
    for e in errs:
        print(" -", e)
    raise RuntimeError("Could not load the model -- see errors above.")
model.eval()

# Some chat templates accept a separate system turn, some do not. Detect it.
try:
    tokenizer.apply_chat_template([{"role": "system", "content": "x"},
                                   {"role": "user", "content": "y"}],
                                  tokenize=False, add_generation_prompt=True)
    SUPPORTS_SYSTEM_ROLE = True
except Exception:
    SUPPORTS_SYSTEM_ROLE = False
print(f"System role supported: {SUPPORTS_SYSTEM_ROLE}")

In [ ]:
# ========== DOWNLOAD THE DATA ==========
import json, requests
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

BASE = "https://raw.githubusercontent.com/sazani/OntologyRAG-Q/main"
ALL_BOOKS = [
    "aashoor_v2.xlsx", "alaloosi_v2.xlsx", "almawirdee_v2.xlsx", "almuyassar_v2.xlsx",
    "alrazi_v2.xlsx", "altasheel_v2.xlsx", "aysaraAltafasir_v2.xlsx", "fathAlqadeer_v2.xlsx",
    "fathaAlbayan_v2.xlsx", "katheer_v2.xlsx", "mukhtasar_v2.xlsx", "qurtubi_v2.xlsx",
    "saadi_v2.xlsx", "tabari_v2.xlsx", "zadAlmaseer_v2.xlsx",
]
# The two books the reference answers were written from.
SOURCE_BOOKS = ["aysaraAltafasir_v2.xlsx", "almuyassar_v2.xlsx"]

BOOK_FILES = ALL_BOOKS if CORPUS == "all" else SOURCE_BOOKS
CHUNKS_PATH = f"chunks_{CORPUS}.json"
INDEX_PATH = f"index_{CORPUS}.faiss"

os.makedirs("books", exist_ok=True)
for fname in BOOK_FILES:
    p = os.path.join("books", fname)
    if not os.path.exists(p):
        r = requests.get(f"{BASE}/Resources/Tafaser/Tafaser_DS2/{fname}")
        r.raise_for_status()
        open(p, "wb").write(r.content)

if not os.path.exists("OntologyQA_v1.json"):
    r = requests.get(f"{BASE}/Resources/OntologyQA_v1.json")
    r.raise_for_status()
    open("OntologyQA_v1.json", "wb").write(r.content)

with open("OntologyQA_v1.json", encoding="utf-8") as f:
    qa_data = json.load(f)

chunks = index = embed_model = None
if os.path.exists(CHUNKS_PATH) and os.path.exists(INDEX_PATH):
    with open(CHUNKS_PATH, encoding="utf-8") as f:
        chunks = json.load(f)
    index = faiss.read_index(INDEX_PATH)
    embed_model = SentenceTransformer(EMBED_MODEL_ID)
    print(f"Cached: {len(chunks)} chunks, {index.ntotal} vectors.")
else:
    print(f"{len(BOOK_FILES)} books ready, {len(qa_data)} QA pairs. Building index below.")

In [ ]:
# ========== AYAT-ONTOLOGY CHUNKING ==========
# One chunk per verse (or verse range), with the ontology fields -- surah name,
# surah number, verse range -- written into the chunk text, as the paper describes.
#
# NOTE: aysaraAltafasir_v2.xlsx uses different column names from the other 14
# books. Handling only the majority schema silently drops all 1,290 of its rows,
# and that book is the source of ~90% of the Direct questions' answers. Both
# schemas are handled here. With CORPUS="all" this gives 55,471 chunks, which
# is exactly the total reported in the paper's Table 6.
import pandas as pd

STANDARD = {"surah": "SURA_num", "start": "Verse_Number_start",
            "end": "Verse_Number_end", "verse": "passages", "tafsir": "Tafsir"}
AYSARA = {"surah": "SURA_num", "aya_range": "AYA_num",
          "verse": "Ayah", "tafsir": "Tafsir"}
SCHEMAS = {"aysaraAltafasir_v2.xlsx": AYSARA}


def _parse_aya_range(v):
    """'217-218' -> (217, 218); '14-15-16' -> (14, 16); '7' -> (7, 7)."""
    nums = [int(p) for p in str(v).split("-") if p.strip().isdigit()]
    return (min(nums), max(nums)) if nums else (None, None)


if chunks is None:
    surah_names = {}
    for d in qa_data:
        sid, nm = d.get("Sura_ID"), d.get("SURA_name")
        if sid is not None and nm:
            try:
                surah_names[int(sid)] = nm
            except (ValueError, TypeError):
                pass

    chunks, per_book = [], {}
    for fname in BOOK_FILES:
        src = fname.replace("_v2.xlsx", "")
        sc = SCHEMAS.get(fname, STANDARD)
        df = pd.read_excel(os.path.join("books", fname))
        made = 0

        for _, row in df.iterrows():
            su = row.get(sc["surah"])
            if pd.isna(su):
                continue
            if "aya_range" in sc:
                st, en = _parse_aya_range(row.get(sc["aya_range"]))
            else:
                st, en = row.get(sc["start"]), row.get(sc["end"])
            if st is None or en is None or pd.isna(st) or pd.isna(en):
                continue

            tafsir = str(row.get(sc["tafsir"], "")).strip()
            if not tafsir or tafsir == "nan":
                continue
            verse = str(row.get(sc["verse"], "")).strip()

            su, st, en = int(su), int(st), int(en)
            sname = surah_names.get(su, f"سورة {su}")
            chunks.append({
                "surah_number": su, "surah_name": sname,
                "start_ayah": st, "end_ayah": en, "source": src,
                "chunk_text": (f"سورة: {sname} (رقم {su})\n"
                               f"الآية رقم {st} إلى {en}: {verse}\n"
                               f"التفسير ({src}): {tafsir}"),
            })
            made += 1

        per_book[src] = made
        print(f"{src:20} {len(df):5} rows -> {made:5} chunks")

    empty = [b for b, m in per_book.items() if m == 0]
    if empty:
        raise RuntimeError(f"No chunks produced from {empty} -- column schema mismatch.")

    with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)

    print(f"\nTOTAL: {len(chunks)} chunks", end="  ")
    if CORPUS == "all":
        print("-- matches the paper's Table 6 total (55,471)." if len(chunks) == 55471
              else "-- expected 55,471 per the paper's Table 6; check before trusting results.")
    else:
        print("(source books only)")
else:
    print(f"Skipped -- {len(chunks)} chunks cached.")

In [ ]:
# ========== BUILD THE SEARCH INDEX (the slow step) ==========
# normalize_embeddings + IndexFlatIP together == cosine similarity,
# which is the "Similarity" search in the paper's best row.
if index is None:
    embed_model = SentenceTransformer(EMBED_MODEL_ID)
    # e5 models need "passage: " on stored text and "query: " on searches.
    texts = ["passage: " + c["chunk_text"] for c in chunks]
    print(f"Embedding {len(texts)} chunks. This is the slow part -- do not interrupt.")
    emb = np.array(embed_model.encode(texts, batch_size=64, show_progress_bar=True,
                                      normalize_embeddings=True), dtype="float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    faiss.write_index(index, INDEX_PATH)
    print("Index built:", index.ntotal, "chunks. Cached for next time.")
else:
    print(f"Skipped -- {index.ntotal} vectors cached.")

In [ ]:
# ========== ENGLISH FOR EVERYTHING ARABIC ==========
# Two separate jobs:
#   1. Surah names -- a fixed lookup, so retrieval printouts are readable instantly.
#   2. Free Arabic text (questions, references, answers) -- translated on demand by
#      the same Gemma model. Used for DISPLAY ONLY; scoring never sees a translation.
import re

SURAH_EN = {
    1: "Al-Fatihah", 2: "Al-Baqarah", 3: "Aal-Imran", 4: "An-Nisa", 5: "Al-Ma'idah",
    6: "Al-An'am", 7: "Al-A'raf", 8: "Al-Anfal", 9: "At-Tawbah", 10: "Yunus",
    11: "Hud", 12: "Yusuf", 13: "Ar-Ra'd", 14: "Ibrahim", 15: "Al-Hijr",
    16: "An-Nahl", 17: "Al-Isra", 18: "Al-Kahf", 19: "Maryam", 20: "Ta-Ha",
    21: "Al-Anbiya", 22: "Al-Hajj", 23: "Al-Mu'minun", 24: "An-Nur", 25: "Al-Furqan",
    26: "Ash-Shu'ara", 27: "An-Naml", 28: "Al-Qasas", 29: "Al-Ankabut", 30: "Ar-Rum",
    31: "Luqman", 32: "As-Sajdah", 33: "Al-Ahzab", 34: "Saba", 35: "Fatir",
    36: "Ya-Sin", 37: "As-Saffat", 38: "Sad", 39: "Az-Zumar", 40: "Ghafir",
    41: "Fussilat", 42: "Ash-Shura", 43: "Az-Zukhruf", 44: "Ad-Dukhan", 45: "Al-Jathiyah",
    46: "Al-Ahqaf", 47: "Muhammad", 48: "Al-Fath", 49: "Al-Hujurat", 50: "Qaf",
    51: "Adh-Dhariyat", 52: "At-Tur", 53: "An-Najm", 54: "Al-Qamar", 55: "Ar-Rahman",
    56: "Al-Waqi'ah", 57: "Al-Hadid", 58: "Al-Mujadilah", 59: "Al-Hashr", 60: "Al-Mumtahanah",
    61: "As-Saff", 62: "Al-Jumu'ah", 63: "Al-Munafiqun", 64: "At-Taghabun", 65: "At-Talaq",
    66: "At-Tahrim", 67: "Al-Mulk", 68: "Al-Qalam", 69: "Al-Haqqah", 70: "Al-Ma'arij",
    71: "Nuh", 72: "Al-Jinn", 73: "Al-Muzzammil", 74: "Al-Muddaththir", 75: "Al-Qiyamah",
    76: "Al-Insan", 77: "Al-Mursalat", 78: "An-Naba", 79: "An-Nazi'at", 80: "Abasa",
    81: "At-Takwir", 82: "Al-Infitar", 83: "Al-Mutaffifin", 84: "Al-Inshiqaq", 85: "Al-Buruj",
    86: "At-Tariq", 87: "Al-A'la", 88: "Al-Ghashiyah", 89: "Al-Fajr", 90: "Al-Balad",
    91: "Ash-Shams", 92: "Al-Layl", 93: "Ad-Duha", 94: "Ash-Sharh", 95: "At-Tin",
    96: "Al-Alaq", 97: "Al-Qadr", 98: "Al-Bayyinah", 99: "Az-Zalzalah", 100: "Al-Adiyat",
    101: "Al-Qari'ah", 102: "At-Takathur", 103: "Al-Asr", 104: "Al-Humazah", 105: "Al-Fil",
    106: "Quraysh", 107: "Al-Ma'un", 108: "Al-Kawthar", 109: "Al-Kafirun", 110: "An-Nasr",
    111: "Al-Masad", 112: "Al-Ikhlas", 113: "Al-Falaq", 114: "An-Nas",
}

_ARABIC_CHAR = re.compile(r"[\u0600-\u06FF]")


def arabic_ratio(s):
    """Share of the alphabetic characters that are Arabic. 0.0 for pure English."""
    letters = [ch for ch in (s or "") if ch.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for ch in letters if _ARABIC_CHAR.match(ch)) / len(letters)


def surah_en(num, fallback=""):
    return SURAH_EN.get(int(num), fallback or f"Surah {num}")


def fmt_chunk_en(c):
    """'[katheer] Al-Fatihah (1) 1-1' -- no Arabic left in retrieval printouts."""
    return (f"[{c['source']}] {surah_en(c['surah_number'])} "
            f"({c['surah_number']}) {c['start_ayah']}-{c['end_ayah']}")


print(f"{len(SURAH_EN)} surah names mapped to English.")
print("Example:", fmt_chunk_en(chunks[0]))

In [ ]:
# ========== PROMPTS: RAG, AND RAG + CHAIN-OF-THOUGHT ==========
# The base prompt is copied word for word from the paper, section 4.
SYSTEM_PROMPT_BASE = """You are an expert in interpreting the Quran, specifically
designed to answer users' questions. Provide answers solely based on the
context provided below. Do not draw upon any external or prior knowledge or
information. If the answer is not found within the given context, respond
with 'I don't know.' Ensure that the Ayah (verses) are quoted verbatim as
they appear in the Quran. All answers should be provided in Arabic."""

# The CoT block is the only addition. Two things it has to get right:
#   1. The reasoning must be SEPARABLE, or it gets scored as part of the answer
#      and destroys BLEU. Hence the mandatory ANSWER: delimiter.
#   2. The answer must stay Arabic, because the references are Arabic.
COT_BLOCK = """

Think step by step before you answer. Reply in exactly this format, and nothing else:

REASONING:
Numbered steps, in English.
1. Which of the numbered context passages are relevant to the question, and which
   are not. Say so explicitly.
2. What each relevant passage states.
3. How those statements combine into the answer, and whether the context is in
   fact sufficient to answer at all.

ANSWER:
The final answer, in Arabic only. Answer the question directly -- no preamble, no
restating the question, no reasoning, no mention of the passages. Quote any Ayah
verbatim. If the reasoning showed the context does not answer the question, write
exactly: I don't know

Write nothing after the Arabic answer."""

SYSTEM_PROMPT = SYSTEM_PROMPT_BASE + (COT_BLOCK if USE_COT else "")

# ---- Pulling the answer back out of a CoT generation ----
# This is the part that quietly breaks a CoT run if you get it wrong: if the split
# fails and the English reasoning is scored as the answer, every metric collapses
# and it looks like the model got worse. Every prediction carries a parse_ok flag,
# and the scoring cell reports how many failed.
# The label may be written as ANSWER:, **ANSWER:**, ### ANSWER, __ANSWER__, or
# the Arabic equivalents. Accept a colon OR a label sitting alone on its line,
# but nothing looser -- "answer" mid-sentence in the reasoning must not match.
_LABEL = r"(?:FINAL\s+ANSWER|ANSWER|الجواب|الإجابة)"
_ANSWER_RE = re.compile(
    r"(?:^|\n)[ \t]*[*_#]*[ \t]*" + _LABEL +
    r"[ \t]*[*_]*[ \t]*(?::[ \t]*[*_]*[ \t]*|(?=\n))",
    re.IGNORECASE)
_REASON_RE = re.compile(
    r"^[ \t]*[*_#]*[ \t]*(?:REASONING|التفكير)"
    r"[ \t]*[*_]*[ \t]*:?[ \t]*",
    re.IGNORECASE)


def split_cot(raw):
    """raw generation -> (reasoning, answer, parse_ok).

    parse_ok is False when the ANSWER: delimiter was missing and we had to guess.
    """
    if not USE_COT:
        return "", raw.strip(), True

    matches = list(_ANSWER_RE.finditer(raw))
    if matches:
        m = matches[-1]                       # last, in case the word appears in the reasoning
        reasoning = _REASON_RE.sub("", raw[:m.start()]).strip()
        return reasoning, raw[m.end():].strip(), True

    # No delimiter -- usually means generation hit MAX_NEW_TOKENS mid-reasoning.
    # Best guess: the trailing run of predominantly-Arabic paragraphs.
    paras = [p.strip() for p in re.split(r"\n\s*\n", raw) if p.strip()]
    arabic_paras = [p for p in paras if arabic_ratio(p) > 0.5]
    if arabic_paras:
        return _REASON_RE.sub("", raw).strip(), "\n\n".join(arabic_paras), False
    return _REASON_RE.sub("", raw).strip(), "", False


print("Prompt in use:", "RAG + CoT" if USE_COT else "RAG (direct answer)")
print("-" * 72)
print(SYSTEM_PROMPT)

In [ ]:
# ========== RETRIEVAL + GENERATION ==========
def retrieve(question, k=TOP_K):
    qv = np.array(embed_model.encode(["query: " + question], normalize_embeddings=True),
                  dtype="float32")
    _, ids = index.search(qv, k)
    return [chunks[i] for i in ids[0]]


def _chat_text(user_prompt, system=SYSTEM_PROMPT):
    msgs = ([{"role": "system", "content": system},
             {"role": "user", "content": user_prompt}] if SUPPORTS_SYSTEM_ROLE else
            [{"role": "user", "content": f"{system}\n\n{user_prompt}"}])
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def _assemble(top, question, chars, system):
    # Numbering the passages gives the CoT something concrete to refer back to
    # in step 1, instead of vaguely gesturing at "the context".
    parts = []
    for i, c in enumerate(top, 1):
        t = c["chunk_text"]
        parts.append(f"[{i}] " + (t[:chars] + " ..." if len(t) > chars else t))
    return _chat_text(f"Context:\n{chr(10).join(parts)}\nQuestion: {question}\nAnswer:",
                      system=system)


def _run(text, max_new_tokens):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    torch.cuda.empty_cache()
    with torch.no_grad():
        # do_sample=False is greedy decoding -- the paper's temperature=0.0.
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()


def generate(top, question, system=None, max_new_tokens=None):
    system = SYSTEM_PROMPT if system is None else system
    max_new_tokens = MAX_NEW_TOKENS if max_new_tokens is None else max_new_tokens

    # Shrink the CONTEXT until the prompt fits, so the question is never cut off --
    # it sits after the context, so truncating the prompt would delete it.
    chars = MAX_CHARS_PER_CHUNK
    for c in [MAX_CHARS_PER_CHUNK, 1200, 800, 500, 300, 150]:
        text = _assemble(top, question, c, system)
        chars = c
        if len(tokenizer(text)["input_ids"]) <= MAX_PROMPT_TOKENS:
            break

    for _ in range(3):
        try:
            return _run(text, max_new_tokens)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            chars = max(150, chars // 2)
            text = _assemble(top, question, chars, system)
            print(f"    [out of memory -- retrying with {chars} chars per chunk]")
    raise RuntimeError("Out of memory even at the smallest context size.")


def translate_to_english(text, max_new_tokens=400):
    """Display only. Never used for scoring."""
    text = (text or "").strip()
    if not text or arabic_ratio(text) < 0.15:
        return text                            # already English, or nothing to do
    prompt = ("Translate the following Arabic text into clear English. "
              "Output only the translation and nothing else.\n\n" + text)
    try:
        return _run(_chat_text(prompt, system="You are a precise Arabic-to-English translator."),
                    max_new_tokens).strip()
    except Exception as e:
        return f"[translation failed: {type(e).__name__}]"


def show_ar(text, label=None, indent="  ", translate=True):
    """Print Arabic as-is, then its English translation on '# EN:' comment lines.

    Every Arabic string this notebook prints goes through here. The Arabic is the
    real output -- the English underneath is only so you can read it.
    """
    if label:
        print(label)
    text = (text or "").strip()
    if not text:
        print(indent + "(empty)")
        return
    for line in text.splitlines():
        print(indent + line)
    if translate and arabic_ratio(text) >= 0.15:
        for line in translate_to_english(text).splitlines():
            if line.strip():
                print(indent + "# EN: " + line.strip())


def ask(question, verbose=True, translate=None, show_passages=False):
    translate = verbose if translate is None else translate
    top = retrieve(question)
    if verbose:
        print("RETRIEVED PASSAGES (k=%d)" % len(top))
        for i, c in enumerate(top, 1):
            print(f"  [{i}] {fmt_chunk_en(c)}")
            if show_passages:
                snippet = c["chunk_text"][:400]
                show_ar(snippet, indent="      ", translate=translate)

    raw = generate(top, question)
    reasoning, answer, parse_ok = split_cot(raw)

    # A CoT run that produced reasoning but no answer scores as an empty string,
    # which is silently ruinous. Fall back to one direct, non-CoT attempt.
    if not answer.strip():
        answer = generate(top, question, system=SYSTEM_PROMPT_BASE, max_new_tokens=512).strip()
        parse_ok = False

    rec = {"prediction": answer, "reasoning": reasoning, "raw": raw, "parse_ok": parse_ok,
           "retrieved": [fmt_chunk_en(c) for c in top]}

    if verbose:
        if reasoning:
            print("\nCHAIN OF THOUGHT (English, not scored)")
            for line in reasoning.splitlines():
                if line.strip():
                    print("  " + line.strip())
        show_ar(answer, label="\nANSWER (Arabic -- this is the output that gets scored)",
                translate=translate)
    return rec


direct_questions = [d for d in qa_data if d.get("Task") == "Direct" and d.get("Answer")]
print(f"Direct questions available: {len(direct_questions)}\n")
print("=" * 72)
print("ONE TEST QUESTION")
print("=" * 72)
_q = direct_questions[0]["Question"]
show_ar(_q, label="QUESTION (Arabic)")
print()
_demo = ask(_q, verbose=True, show_passages=TRANSLATE_PASSAGES)
show_ar(direct_questions[0]["Answer"],
        label="\nREFERENCE ANSWER (Arabic -- the gold answer it is scored against)")

In [ ]:
# ========== ANSWER THE QUESTIONS ==========
# Saved after every question. If this stops early, just run it again --
# it continues from where it left off instead of starting over.
#
# The sample is always drawn at the FULL size and then sliced, so the QUICK_TEST
# questions are the first 5 of the real 500. Both write to the same file, so the
# full run reuses the quick-test answers instead of redoing them.
import random, time
from tqdm.auto import tqdm

model_tag = MODEL_ID.split("/")[-1]
path = os.path.join(RESULTS_DIR,
                    f"preds_{model_tag}_{CORPUS}_{RUN_TAG}_n{N_EVAL_SAMPLES}_seed{SEED}.json")

random.seed(SEED)
full_sample = (list(direct_questions) if N_EVAL_SAMPLES is None else
               random.sample(direct_questions, min(N_EVAL_SAMPLES, len(direct_questions))))
sample = full_sample[:N_ANSWER]

done = {}
if os.path.exists(path):
    done = {r["question"]: r for r in json.load(open(path, encoding="utf-8"))}
    print(f"Resuming: {len(done)} already answered, {len(sample)} wanted.")

records, errors, t0 = [], 0, time.time()
for item in tqdm(sample, desc=f"answering ({RUN_TAG})"):
    q = item["Question"]
    if q in done:
        records.append(done[q])
        continue
    try:
        res = ask(q, verbose=False)
    except Exception as e:
        # One bad question must not kill a multi-hour run. Record it honestly as an
        # empty prediction -- dropping it instead would quietly inflate the scores.
        errors += 1
        print(f"\n[error on a question: {type(e).__name__}: {str(e)[:120]}]")
        res = {"prediction": "", "reasoning": "", "raw": "",
               "parse_ok": False, "retrieved": [], "error": f"{type(e).__name__}"}
    rec = {"question": q, "reference": item["Answer"]}
    rec.update(res)
    records.append(rec)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=1)

mins = (time.time() - t0) / 60
print(f"\n{len(records)} answers saved to {path}")
print(f"Elapsed: {mins:.1f} min", end="")
new = sum(1 for r in records if r["question"] not in done)
if new:
    left = max(0, len(full_sample) - len(records))
    print(f"  ({mins*60/new:.1f} s per new answer -- about "
          f"{mins/new*left/60:.1f} h for the remaining {left})")
else:
    print()
if errors:
    print(f"{errors} question(s) errored and are stored as empty predictions.")

In [ ]:
# ========== THE RESULT ==========
import sacrebleu
from bert_score import score as bert_score

preds = [r["prediction"] for r in records]
refs = [r["reference"] for r in records]

# --- health checks first: these decide whether the scores mean anything ---
n_empty = sum(1 for p in preds if not p.strip())
n_badparse = sum(1 for r in records if not r.get("parse_ok", True))
n_english = sum(1 for p in preds if p.strip() and arabic_ratio(p) < 0.5)

print("Sanity checks")
print("-" * 72)
print(f"  empty predictions        {n_empty:4d} / {len(preds)}"
      + ("   <-- these score 0" if n_empty else ""))
print(f"  CoT answer-split failed  {n_badparse:4d} / {len(preds)}"
      + ("   <-- answer was guessed, not delimited" if n_badparse else ""))
print(f"  answered in English      {n_english:4d} / {len(preds)}"
      + ("   <-- references are Arabic, so these score ~0" if n_english else ""))
print()

bleu = sacrebleu.corpus_bleu(preds, [refs]).score
# The default '13a' tokenizer is built for European text and understates Arabic.
# Reported as a supplement; the 13a number stays primary for paper comparison.
bleu_intl = sacrebleu.corpus_bleu(preds, [refs], tokenize="intl").score
chrf = sacrebleu.corpus_chrf(preds, [refs]).score
P, R, F1 = bert_score(preds, refs, model_type=BERTSCORE_MODEL, verbose=False)
p, r, f1 = P.mean().item()*100, R.mean().item()*100, F1.mean().item()*100

pred_len = sum(len(x) for x in preds) / max(1, len(preds))
ref_len = sum(len(x) for x in refs) / max(1, len(refs))

print("=" * 72)
print(f"  OntologyRAG-Q best configuration + {'CoT' if USE_COT else 'direct'} "
      f"— {MODEL_ID}")
print("=" * 72)
print(f"  BLEU             {bleu:6.2f}   (sacrebleu default '13a' tokenizer)")
print(f"  BLEU (intl)      {bleu_intl:6.2f}   (supplementary, fairer to Arabic)")
print(f"  CHRF             {chrf:6.2f}")
print(f"  BERT Precision   {p:6.2f}")
print(f"  BERT Recall      {r:6.2f}")
print(f"  BERT F1          {f1:6.2f}")
print("=" * 72)
print(f"  {len(records)} questions · {QUANT_BITS}-bit · {len(chunks)} chunks "
      f"· corpus={CORPUS} · prompting={RUN_TAG} · seed {SEED}")
print(f"  BERTScore model: {BERTSCORE_MODEL}")
print(f"  Mean length: prediction {pred_len:.0f} chars vs reference {ref_len:.0f} chars"
      f"  ({pred_len/max(1,ref_len):.1f}x)")
if pred_len > 2 * ref_len:
    print("  ^ Answers much longer than the references. BLEU and CHRF punish that")
    print("    hard regardless of correctness -- read the worked examples below")
    print("    before concluding the answers are wrong.")
print()

print("Table 4 row:")
print(f"{'LLM':<16}{'Chunk':<16}{'Parameters':<40}{'Embed':<10}"
      f"{'BLEU':>7}{'CHRF':>7}{'Prec':>7}{'Rec':>7}{'F1':>7}")
print("-" * 117)
params = f"k:6, Similarity, temperature=0.0{', CoT' if USE_COT else ''}"
print(f"{model_tag:<16}{'Ayat ontology':<16}{params:<40}{'E5-small':<10}"
      f"{bleu:>7.2f}{chrf:>7.2f}{p:>7.2f}{r:>7.2f}{f1:>7.2f}")

json.dump({"model": MODEL_ID, "quant_bits": QUANT_BITS, "corpus": CORPUS,
           "prompting": RUN_TAG, "use_cot": USE_COT,
           "n": len(records), "seed": SEED, "n_chunks": len(chunks),
           "bertscore_model": BERTSCORE_MODEL,
           "n_empty": n_empty, "n_parse_failed": n_badparse, "n_english": n_english,
           "mean_pred_chars": pred_len, "mean_ref_chars": ref_len,
           "BLEU": bleu, "BLEU_intl": bleu_intl, "CHRF": chrf,
           "BERT_P": p, "BERT_R": r, "BERT_F1": f1},
          open(os.path.join(RESULTS_DIR,
                            f"result_{model_tag}_{CORPUS}_{RUN_TAG}_n{len(records)}.json"),
               "w"), indent=2)
print("\nSaved.")

In [ ]:
# ========== WORKED EXAMPLES, IN ENGLISH ==========
# This is the cell that tells you whether the run is actually working. The metrics
# above cannot distinguish "wrong answer" from "right answer, phrased differently
# and three times too long" -- reading a few examples can.
#
# Everything Arabic is shown with its English translation underneath.
n_show = min(TRANSLATE_N, len(records))
print(f"Translating {n_show} example(s) with {MODEL_ID}. "
      f"Raise TRANSLATE_N for more (each costs a generation).\n")

report = []
for i, rec in enumerate(records[:n_show], 1):
    print("=" * 78)
    print(f"EXAMPLE {i} of {n_show}")
    print("=" * 78)

    q_en = translate_to_english(rec["question"])
    ref_en = translate_to_english(rec["reference"])
    pred_en = (translate_to_english(rec["prediction"])
               if rec["prediction"].strip() else "(empty)")
    report.append({**rec, "question_en": q_en, "reference_en": ref_en,
                   "prediction_en": pred_en})

    # show_ar would re-translate; reuse what we already paid for above.
    def _show(text, en, label, indent="  "):
        print(label)
        for line in (text or "(empty)").splitlines():
            print(indent + line)
        for line in (en or "").splitlines():
            if line.strip():
                print(indent + "# EN: " + line.strip())

    _show(rec["question"], q_en, "\nQUESTION (Arabic)")

    if rec.get("retrieved"):
        print("\nRETRIEVED PASSAGES (k=%d)" % len(rec["retrieved"]))
        for j, ref in enumerate(rec["retrieved"], 1):
            print(f"  [{j}] {ref}")

    if rec.get("reasoning"):
        print("\nCHAIN OF THOUGHT (English, not scored)")
        for line in rec["reasoning"].splitlines():
            if line.strip():
                print("  " + line.strip())

    _show(rec["prediction"], pred_en,
          "\nMODEL ANSWER (Arabic -- this is what gets scored)"
          + ("" if rec.get("parse_ok", True) else "   [!] answer-split failed"))

    _show(rec["reference"], ref_en,
          "\nREFERENCE ANSWER (Arabic -- the gold answer)")

    print(f"\nLENGTH  model {len(rec['prediction'])} chars vs reference "
          f"{len(rec['reference'])} chars")
    print()

out = os.path.join(RESULTS_DIR, f"examples_en_{model_tag}_{CORPUS}_{RUN_TAG}.json")
json.dump(report, open(out, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("=" * 78)
print(f"Saved {len(report)} translated examples to {out}")
print()
print("What to check before setting QUICK_TEST = False:")
print("  1. Do the retrieved passages come from the surah the question is about?")
print("     If not, retrieval is the bottleneck, not the model.")
print("  2. Does the chain of thought reference the numbered passages, or is it")
print("     inventing? Invented reasoning means the context is not being used.")
print("  3. Is the model answer Arabic, and roughly the length of the reference?")
print("     Much longer answers are the single biggest drag on BLEU and CHRF.")
print("  4. Is 'answer-split failed' clear on all 5? If not, raise MAX_NEW_TOKENS.")